# 📦 实验一：环境配置与模型加载

## 学习目标
- 配置 CANNLab 平台环境（Ascend NPU）
- 下载 DeepSeek-LLM-7B 基座模型
- 加载分词器（Tokenizer）
- 测试模型基本推理能力

## 1. 环境准备

CANNLab 默认镜像已预装 `torch` 与 `torch_npu`，下方 cell 仅安装训练所需的上层依赖。

> **首次运行**：直接执行下方 cell 即可，安装完成后无需重启 kernel。

In [ ]:
# 1. 安装训练所需依赖（使用更稳定的版本）
%pip install -q -i https://pypi.tuna.tsinghua.edu.cn/simple modelscope==1.35.4 transformers==5.5.4 trl==1.2.0 datasets==4.8.4 swanlab==0.7.15 accelerate==1.13.0 peft bitsandbytes 

# 2. 注册 Ascend NPU 后端
import torch
import torch_npu  # noqa: F401

print(f"torch: {torch.__version__}")
print(f"NPU 可用: {torch.npu.is_available()}, 卡数: {torch.npu.device_count()}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

## 2. 下载模型

从 ModelScope 下载 DeepSeek-LLM-7B-Chat 模型（约 15GB，下载需几分钟）。

In [ ]:
# 下载基座模型（静默模式）
!modelscope download --model deepseek-ai/deepseek-llm-7b-chat --local_dir ./model/deepseek-ai/deepseek-llm-7b-chat 2>/dev/null
print("✅ 模型下载完成")

## 3. 加载模型与分词器（启用 FlashAttention 算子加速）

> **硬件说明**：本实验基于 **Ascend NPU（昇腾神经网络处理器）** 运行，通过 CANNLab 平台提供算力支持。
> 与 GPU 的 CUDA 生态不同，Ascend NPU 使用 `torch_npu` 作为后端驱动。

In [ ]:
from transformers import AutoModelForCausalLM, PreTrainedTokenizerFast

MODEL_PATH = "./model/deepseek-ai/deepseek-llm-7b-chat"

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=f"{MODEL_PATH}/tokenizer.json",
    pad_token="</s>",
    eos_token="</s>",
    bos_token="<s>"
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa"  # 🔥 启用 SDPA 加速注意力计算
)

print(f"模型已加载，设备: {model.device}")
print(f"词汇表大小: {len(tokenizer)}")

---

## 🔥 拓展阅读：FlashAttention 算子加速原理解析

> 本节内容参考自 [AIInfraGuide — FlashAttention V1/V2 详解](https://caomaolufei.github.io/AIInfraGuide/)，深入剖析算子底层原理。

---

### 一、标准 Attention 的性能瓶颈

#### 1.1 计算流程

标准的 Scaled Dot-Product Attention 按以下步骤执行：

$$S = QK^\top / \sqrt{d}, \quad P = \text{softmax}(S), \quad O = PV$$

其中 $Q, K, V \in \mathbb{R}^{N \times d}$，$N$ 是序列长度，$d$ 是 head dimension。

#### 1.2 显存瓶颈

标准实现的关键问题在于中间矩阵 $S$ 和 $P$ 的大小为 $N \times N$：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">指标</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">数值</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">中间矩阵大小</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$O(N^2)$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$S$ 和 $P$ 各需 $N \times N$ 存储</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">HBM 读写量</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$O(N^2)$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">需要反复读写 $N \times N$ 矩阵</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">计算量</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$O(N^2 d)$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">两次矩阵乘法</td>
    </tr>
  </tbody>
</table>

当序列长度 $N=4096$、$d=128$ 时，单个 head 的 $S$ 矩阵就需要 $4096 \times 4096 \times 2 = 32 \text{MB}$（FP16）。对于多 head 和多 batch 的场景，显存开销极为可观。

#### 1.3 算术强度不足

标准 Attention 的 **算术强度（Arithmetic Intensity）** 为：

$$\text{AI} = \frac{O(N^2 d)}{O(N^2 + Nd)} \approx O(d)$$

由于 $d$ 通常只有 64 或 128，算术强度较低，**运算瓶颈在于显存带宽而非算力**。这意味着 GPU 大量时间花在数据搬运上，计算单元处于饥饿状态。

> 💡 **类比**：就像一个厨师做菜的速度很快，但食材从仓库搬到厨房的速度太慢，导致厨师大部分时间在等食材——标准 Attention 的核心瓶颈就是 **HBM 和 SRAM 之间的数据搬运**。

---

### 二、FlashAttention V1 核心思想

#### 2.1 IO-Awareness：关注数据搬运

FlashAttention（Dao et al., 2022）的核心洞察是：**在现代 GPU 上，Attention 是一个 Memory-Bound 操作**，优化的关键不在于减少浮点运算，而在于减少 HBM 访问次数。

传统优化思路专注于减少 FLOPs（如稀疏 Attention），但 FlashAttention 另辟蹊径——通过精巧的分块计算策略，让数据尽可能留在 **SRAM（Shared Memory）** 中完成计算，避免中间结果落地到 HBM。

#### 2.2 两个关键技术

FlashAttention V1 依赖两个核心技术的组合：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">技术</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>Tiling（分块）</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">将 $Q, K, V$ 切成小块，每次只加载一小块到 SRAM 中计算，避免生成完整的 $N \times N$ 矩阵</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>Online Softmax（在线 Softmax）</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">在不知道全局最大值的情况下，边加载新块边修正 Softmax 结果</td>
    </tr>
  </tbody>
</table>

> 📌 **关键点**：这两个技术缺一不可——Tiling 解决了"如何不生成完整的 $N \times N$ 矩阵"的问题，Online Softmax 解决了"分块计算时 Softmax 依赖全局信息"的问题。

#### 2.3 直觉类比

想象你在做一道需要全班考试成绩来计算每个人排名百分比的问题。传统方法是先收齐所有人的分数（全部写在黑板上），再统一计算。FlashAttention 的做法相当于：**每收到一组学生的分数，就立刻更新当前的统计量（最大值、求和），并修正之前的计算结果**——最终得到的答案和收齐后统一计算完全相同。

---

### 三、Online Softmax 算法详解

#### 3.1 标准 Softmax 的三遍扫描

为保证数值稳定性，标准（safe）Softmax 需要三遍扫描输入向量 $x \in \mathbb{R}^N$：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">遍数</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">计算</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">第一遍</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$m = \max_{i} x_i$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">求全局最大值</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">第二遍</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$\ell = \sum_j e^{x_j - m}$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">计算归一化分母</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">第三遍</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$\text{softmax}(x_i) = e^{x_i - m} / \ell$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">计算每个元素的输出</td>
    </tr>
  </tbody>
</table>

第二、三遍都依赖第一遍得到的全局最大值 $m$，第三遍又依赖第二遍得到的 $\ell$，这要求我们必须先完整扫描一次才能开始下一遍，显然与分块策略矛盾。

#### 3.2 Online Softmax 的递推公式

Online Softmax 的关键创新在于：**维护一个可修正的局部统计量，随着新数据块的到来逐步更新**。

假设我们已经处理了前 $j-1$ 块数据，维护了局部最大值 $m^{(j-1)}$ 和局部指数和 $l^{(j-1)}$。当第 $j$ 块数据 $x^{(j)}$ 到来时：

**步骤 1**：更新最大值
$$m^{(j)} = \max(m^{(j-1)}, \max(x^{(j)}))$$

**步骤 2**：修正并更新指数和
$$l^{(j)} = l^{(j-1)} \cdot e^{m^{(j-1)} - m^{(j)}} + \sum_i e^{x^{(j)}_i - m^{(j)}}$$

**步骤 3**：修正已有输出
$$O^{(j)} = O^{(j-1)} \cdot \frac{l^{(j-1)} \cdot e^{m^{(j-1)} - m^{(j)}}}{l^{(j)}} + \frac{\sum_i e^{x^{(j)}_i - m^{(j)}} \cdot V^{(j)}_i}{l^{(j)}}$$

#### 3.3 正确性保证

Online Softmax 的数学正确性来源于一个简单的恒等式：

$$e^{x_i - m_{\text{new}}} = e^{x_i - m_{\text{old}}} \cdot e^{m_{\text{old}} - m_{\text{new}}}$$

每当全局最大值更新时，之前所有的指数项都可以通过乘以一个修正因子 $e^{m_{\text{old}} - m_{\text{new}}}$ 来得到正确结果。**这个操作是精确的，没有任何近似。**

> ⚠️ **注意**：Online Softmax 产生的结果与标准 Softmax 在数学上完全等价（除了浮点运算顺序不同可能带来的微小舍入差异），**这不是一种近似算法**。

---

### 四、Tiling 分块计算策略

#### 4.1 块大小选择

将 $Q$ 分成 $T_r = \lceil N / B_r \rceil$ 个块，$K, V$ 分成 $T_c = \lceil N / B_c \rceil$ 个块。

块大小取决于 **SRAM 容量 $M$**（单位：元素个数）和注意力头维度 $d$：

$$B_c = \left\lceil \frac{M}{4d} \right\rceil, \quad B_r = \min\left(\left\lceil \frac{M}{4d} \right\rceil, d\right)$$

**为什么是 $M/4d$？** 计算一个块时，SRAM 需要同时驻留 $Q_i, K_j, V_j, O_i$ 四块大小约为 $B \times d$ 的张量，合计约 $4Bd$ 个元素。要求 $4Bd \le M$，即可解出每块最多放 $M/4d$ 行。

#### 4.2 SRAM 使用规划

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">数据</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">大小</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">用途</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$Q_i$ 块</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$B_r \times d$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">当前 Query 块</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$K_j$ 块</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$B_c \times d$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">当前 Key 块</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$V_j$ 块</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$B_c \times d$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">当前 Value 块</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$S_{ij}$ 块</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$B_r \times B_c$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">局部注意力分数</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$O_i$ 块</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$B_r \times d$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">累积输出块</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$m_i, l_i$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$2 B_r$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">行最大值、行指数和</td>
    </tr>
  </tbody>
</table>

#### 4.3 V1 的循环结构

FlashAttention V1 采用 **外循环遍历 K/V 块，内循环遍历 Q 块** 的策略：

```
外循环: for j = 1 to T_c (遍历 K/V 块)
    从 HBM 加载 K_j, V_j 到 SRAM
    内循环: for i = 1 to T_r (遍历 Q 块)
        从 HBM 加载 Q_i, O_i, m_i, l_i 到 SRAM
        计算局部注意力分数 S_ij = Q_i * K_j^T / sqrt(d)
        更新统计量和输出（Online Softmax）
        将更新后的 O_i, m_i, l_i 写回 HBM
```

#### 4.4 前向传播伪代码

```
def flash_attention_v1_forward(Q, K, V, B_r, B_c):
    O = zeros(N, d)
    m = full(N, -inf)     # 行最大值
    l = zeros(N)          # 行指数和

    for j in range(0, N, B_c):          # 外循环：K/V 块
        K_j = K[j:j+B_c]                # 加载到 SRAM
        V_j = V[j:j+B_c]                # 加载到 SRAM

        for i in range(0, N, B_r):      # 内循环：Q 块
            Q_i = Q[i:i+B_r]            # 加载到 SRAM
            O_i, m_i, l_i = O[i:i+B_r], m[i:i+B_r], l[i:i+B_r]

            S_ij = Q_i @ K_j.T / sqrt(d)  # 局部注意力分数
            m_new = max(m_i, rowmax(S_ij))
            l_new = l_i * exp(m_i - m_new) + rowsum(exp(S_ij - m_new))
            O_new = O_i * (l_i * exp(m_i - m_new) / l_new) + (exp(S_ij - m_new) / l_new) @ V_j

            O[i:i+B_r], m[i:i+B_r], l[i:i+B_r] = O_new, m_new, l_new  # 写回 HBM

    return O
```

---

### 五、反向传播与重计算

标准 Attention 的反向传播需要中间矩阵 $S$ 和 $P$（大小为 $N \times N$）来计算梯度。如果保存这些中间矩阵，前向传播省下的显存就白费了。

**FlashAttention 的解决方案**：不保存 $S$ 和 $P$，反向传播时使用分块方式重新计算它们。前向传播只保存 $Q, K, V, O, m, l$（输入、输出和统计量），反向传播时用同样的分块策略重新计算 $S$ 和 $P$ 的局部块，然后计算梯度。

> 这种 **重计算（Recomputation）** 策略本质上是"用计算换显存"——因为重计算只涉及当前块的矩阵乘法，比从 HBM 读回完整 $S$ 矩阵更快。

---

### 六、IO 复杂度分析

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">操作</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">标准 Attention</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">FlashAttention V1</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">提升</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">HBM 访问量</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$O(N^2 d + N^2)$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$O(N^2 d^2 / M)$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">当 $M \gg d$ 时显著减少</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">额外显存</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$O(N^2)$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">$O(N)$</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>平方 → 线性</strong></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">计算速度（实测）</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">1x</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">2.5-3x</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">A100 上实测</td>
    </tr>
  </tbody>
</table>

---

### 七、FlashAttention V2 的主要改进

V1 在 A100 上只达到了理论 FLOPS 的 **25-40%**。V2 通过三项核心改进将其提升到 **50-73%**：

#### 改进 1：调换循环顺序

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">版本</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">外循环</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">内循环</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">问题</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">V1</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">K/V 块</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">Q 块</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">Q 和 O 被反复读写，并行度受限</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>V2</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>Q 块（并行）</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>K/V 块</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">Q 只加载一次，O 只写回一次，天然可并行</td>
    </tr>
  </tbody>
</table>

**V2 的新循环结构**：
```
外循环: for i = 1 to T_r (遍历 Q 块) ← 分配给不同 Thread Block，可并行
    加载 Q_i 到 SRAM（只加载一次）
    初始化 O_i = 0, m_i = -inf, l_i = 0
    内循环: for j = 1 to T_c (遍历 K/V 块)
        加载 K_j, V_j 到 SRAM
        计算 S_ij，更新统计量和 O_i（全在 SRAM 中）
    将最终 O_i 写回 HBM（只写一次）
```

#### 改进 2：减少非矩阵乘运算

V2 将 Softmax 的 rescaling 步骤 **延迟到内循环结束后统一执行**，消除了内循环中每次迭代的 2 次除法 + 1 次乘法：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">版本</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">内循环操作</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">额外开销</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">V1</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">每步都做反归一化 + 重新归一化</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">2 除 + 1 乘</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>V2</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">只维护未归一化的累积，最后统一除一次</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>0</strong>（内循环中）</td>
    </tr>
  </tbody>
</table>

#### 改进 3：Warp 级工作分配优化

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">版本</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">Warp 切分维度</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">问题</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">V1</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">沿 K/V 维度（split-K）</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">多个 Warp 需要合并 Softmax 结果，频繁 Warp 间同步</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>V2</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">沿 Q 维度（split-Q）</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">每个 Warp 独立完成，无需 Warp 间通信</td>
    </tr>
  </tbody>
</table>

#### 性能对比

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">序列长度</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">V1 TFLOPS</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">V2 TFLOPS</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">V2 利用率</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">1024</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">124</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">196</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">63%</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">2048</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">136</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">218</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">70%</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">4096</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">141</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">227</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">73%</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">8192</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">138</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">222</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">71%</td>
    </tr>
  </tbody>
</table>

V2 相比 V1 实现了约 **1.6x 的加速**，在 A100 上达到理论峰值的 50-73%。

---

### 八、在本项目中的实现方式：SDPA

本项目中我们使用 `attn_implementation="sdpa"` 来启用加速：

```python
model = AutoModelForCausalLM.from_pretrained(
    ...,
    attn_implementation="sdpa"  # 使用 PyTorch SDPA
)
```

**SDPA（Scaled Dot-Product Attention）** 是 PyTorch 2.0+ 内置的注意力计算 API（`torch.nn.functional.scaled_dot_product_attention`），它会根据硬件自动选择最优后端：

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">环境</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">自动选择的后端</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>NVIDIA GPU（CUDA）</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">FlashAttention-2 或 Memory Efficient Attention 内核</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>Ascend NPU（本实验）</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">通过 `torch_npu` 集成，自动路由到昇腾的 NPU 融合注意力算子</td>
    </tr>
  </tbody>
</table>

> 💡 **为什么是 `sdpa` 而不是 `flash_attention_2`？**
> - `attn_implementation="flash_attention_2"` 需要安装 `flash-attn` 包，且仅支持 CUDA
> - `attn_implementation="sdpa"` 是 PyTorch 原生方案，**同时支持 NPU 和 CUDA**，跨平台兼容性更好
> - 在 Ascend NPU 上，`torch_npu` 已重载 SDPA 算子，使得 `sdpa` 模式能自动调用 NPU 的硬件加速单元

#### 什么时候受益最大？

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">场景</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">加速效果</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">原因</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">短序列（≤512 tokens）</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">1-2x</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">计算量小，带宽瓶颈不明显</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>中长序列（1024-4096）</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>3-8x</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">注意力矩阵开始占主导</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">长序列（&gt;4096）</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">8-15x</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">标准 Attention 显存溢出，FlashAttention 仍可运行</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>训练（本实验）</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><strong>显存节省 50%+</strong></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">无需保存中间注意力矩阵</td>
    </tr>
  </tbody>
</table>

在本项目的 LoRA 微调中，序列长度设为 1024，FlashAttention 可以显著降低显存压力，让 7B 模型在单卡上更稳定地训练。

---

## 📚 参考资料

- [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135) (NeurIPS 2022)
- [FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691) (ICLR 2024)
- [AIInfraGuide — FlashAttention V1 详解](https://caomaolufei.github.io/AIInfraGuide/guides/模块二-cuda编程与算子优化/61-flashattention-v1详解/)
- [AIInfraGuide — FlashAttention V2 详解](https://caomaolufei.github.io/AIInfraGuide/guides/模块二-cuda编程与算子优化/62-flashattention-v2详解/)

---

## 4. 测试推理

使用 `apply_chat_template` 格式化对话输入（参考 Notebook 的标准做法）。

In [5]:
def chat_infer(model, tokenizer, query, max_new_tokens=256):
    """推理函数：用对话格式包装，只生成一轮回答"""
    prompt = f"<|user|>\n{query}\n<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    input_len = inputs["input_ids"].shape[1]
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    # 只解码新生成的部分，在 <|end|> 处截断
    new_tokens = outputs[0][input_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=False)
    end = answer.find("<|end|>")
    if end != -1:
        answer = answer[:end]
    # 去掉可能生成的 <|user|> 后续内容
    user_pos = answer.find("<|user|>")
    if user_pos != -1:
        answer = answer[:user_pos]
    return answer.strip()

In [ ]:
# 测试几个问题
test_queries = [
    "我总是焦虑不安，压力大得喘不过气，怎么办？",
    "每天提不起劲，觉得自己一无是处，是不是抑郁了？",
]

for q in test_queries:
    print(f"输入：{q}")
    print(f"输出：{chat_infer(model, tokenizer, q)}")
    print("-" * 60)

## 小结

✅ 完成了 CANNLab 环境配置（Ascend NPU）
✅ 成功加载 DeepSeek-LLM-7B 模型
✅ 启用了 **SDPA（FlashAttention）算子加速**——通过 `attn_implementation="sdpa"` 让注意力计算在 NPU 上获得硬件加速
✅ 理解了 FlashAttention 的分块计算与在线 Softmax 原理
✅ 模型推理正常

下一节将探索训练数据的格式和结构。

## 课后练习

1. (单选题) DeepSeek 的 tokenizer 在 transformers 5.x 下 AutoTokenizer 返回空 input_ids，最稳妥的修复是？
   - A. 用 PreTrainedTokenizerFast(tokenizer_file="tokenizer.json") 直接加载
   - B. 换用 GPT2Tokenizer
   - C. 禁用 tokenizer
   - D. 增大 max_length

2. (单选题) device_map="auto" 与 device_map="npu:0" 的关键区别是？
   - A. auto 可能自动跨设备切分，npu:0 强制单卡
   - B. 两者完全相同
   - C. auto 一定更快
   - D. npu:0 会自动均衡多卡

3. (多选题) 加载 6.9B 参数模型时，显存消耗主要来自？
   - A. bf16 权重
   - B. KV cache
   - C. 中间激活
   - D. tokenizer 词表文件

4. (多选题) SDPA 相比 Eager Attention 的优势包括？
   - A. 融合 attention kernel
   - B. 无需额外安装 flash-attn
   - C. 显存复杂度可降至 O(N)（FlashAttention 类实现）
   - D. 所有场景精度完全一致

5. (判断题) attn_implementation="sdpa" 在 NPU 上必须预装 flash-attn 才能运行。

6. (判断题) DeepSeek tokenizer 默认没有可用的 pad_token，加载后应显式设置。

7. (填空题) 6,914,297,856 参数以 bf16 保存，权重占用约 ____ GB。

8. (填空题) AutoModelForCausalLM.from_pretrained 中控制多卡自动切分的参数是 ____，控制低精度加载的参数是 ____。

9. (简答题) 为什么加载 DeepSeek 模型常需要 trust_remote_code=True？

10. (简答题) 如何验证 tokenizer 与 model 已正确加载并真正位于 NPU？

11. (代码设计题) 编写 load_model_and_tokenizer(model_id)，要求 bf16、device_map 可控、打印参数量与 device。

12. (单选题) generate 返回空字符串，优先排查顺序正确的是？
   - A. pad/eos/bos 设置 → 生成参数 → 输入格式
   - B. 显存 → 数据集 → 优化器
   - C. 学习率 → 数据增强
   - D. 模型结构

13. (多选题) 控制生成多样性的参数包括？
   - A. do_sample
   - B. temperature
   - C. top_p
   - D. max_new_tokens

14. (判断题) SDPA 通过 PyTorch 原生 API 在 CUDA 与 NPU 上均可路由到融合注意力实现，是跨平台加速方案。

15. (简答题) 估算 7B bf16 模型在 max_seq_len=2048、batch=1 时 KV cache 的大致开销，并说明计算方式。

> 参考答案见 answer/05.02_environment_and_model_loading_answer.ipynb。